# Day 3 — Milestone 1  
## Review Bronze Outputs

Goal:
- Inspect Bronze tables
- Understand schema
- Identify cleaning needs
- Verify domain separation

In [0]:
import yaml

config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]

raw_schema = config["schemas"]["raw"]
refined_schema = config["schemas"]["refined"]

print("Catalog:", catalog)
print("Raw Schema:", raw_schema)
print("Refined Schema:", refined_schema)

In [0]:
bronze_tables = {
    "market_prices": f"{catalog}.{raw_schema}.bronze_market_prices",
    "weather": f"{catalog}.{raw_schema}.bronze_weather",
    "grid_events": f"{catalog}.{raw_schema}.bronze_grid_events",
    "reference": f"{catalog}.{raw_schema}.bronze_reference"
}

bronze_tables

In [0]:
for name, table in bronze_tables.items():
    try:
        spark.table(table)
        print(f"✅ {name} exists -> {table}")
    except:
        print(f"❌ {name} MISSING -> {table}")

In [0]:
for name, table in bronze_tables.items():
    print(f"\n===== {name.upper()} SCHEMA =====")
    try:
        spark.table(table).printSchema()
    except:
        print("Table not found")

In [0]:
for name, table in bronze_tables.items():
    print(f"\n===== {name.upper()} SAMPLE =====")
    try:
        display(spark.table(table).limit(10))
    except:
        print("Table not found")

In [0]:
for name, table in bronze_tables.items():
    try:
        count = spark.table(table).count()
        print(f"{name}: {count} rows")
    except:
        print(f"{name}: not available")

In [0]:
from pyspark.sql import functions as F

def check_nulls(df):
    return df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])

for name, table in bronze_tables.items():
    print(f"\n===== NULL CHECK: {name} =====")
    try:
        df = spark.table(table)
        display(check_nulls(df))
    except:
        print("Skipped")

## Observations

### market_prices
- check null values
- check numeric columns (price, volume)
- check duplicates

### weather
- check timestamp format
- check missing temperature values

### grid_events
- ensure event_id uniqueness
- standardize severity
- convert timestamps

### reference
- check duplicates
- ensure consistency